In [1]:
import sys, os
if "google.colab" in sys.modules and not os.path.exists(".setup_complete"):
    # Install xvfb and our launcher script for it
    !apt-get install -y xvfb
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/xvfb -O ../xvfb

    # Download dependencies from Github
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/atari_wrappers.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/env_batch.py
    !wget https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/week06_policy_based/runners.py

    # Update the gym environment to be compatible with the Atari environment
    !pip install -q gymnasium[atari,accept-rom-license]
    !pip install -q tensorboardX

    !touch .setup_complete

# This code creates a virtual display to draw game images on.
# It will have no effect if your machine has a monitor.
if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ["DISPLAY"] = ":1"

../xvfb: line 3: $'\r': command not found
../xvfb: line 6: $'\r': command not found
../xvfb: line 21: syntax error near unexpected token `$'in\r''
../xvfb: line 21: `case "$1" in
'


# Implementing Advantage-Actor Critic (A2C)

In this notebook you will implement Advantage Actor Critic algorithm that trains on a batch of Atari 2600 environments running in parallel.

Firstly, we will use environment wrappers implemented in file `atari_wrappers.py`. These wrappers preprocess observations (resize, grayscale, take max between frames, skip frames and stack them together) and rewards. Some of the wrappers help to reset the environment and pass `done` flag equal to `True` when agent dies.
File `env_batch.py` includes implementation of `ParallelEnvBatch` class that allows to run multiple environments in parallel. To create an environment we can use `nature_dqn_env` function. Note that if you are using
PyTorch and not using `tensorboardX` you will need to implement a wrapper that will log **raw** total rewards that the *unwrapped* environment returns and redefine the implemention of `nature_dqn_env` function here.



In [2]:
import numpy as np
import gymnasium as gym
from atari_wrappers import nature_dqn_env


env_name = "SpaceInvadersNoFrameskip-v4"
nenvs = 8  # change this if you have more than 8 CPU ;)
summaries = "Tensorboard"

env = nature_dqn_env(env_name, nenvs=nenvs, summaries=summaries)
obs, _ = env.reset()
assert obs.shape == (nenvs, 4, 84, 84)
assert obs.dtype == np.float32


Next, we will need to implement a model that predicts logits and values. It is suggested that you use the same model as in [Nature DQN paper](https://www.nature.com/articles/nature14236) with a modification that instead of having a single output layer, it will have two output layers taking as input the output of the last hidden layer. **Note** that this model is different from the model you used in homework where you implemented DQN. You can use your favorite deep learning framework here. We suggest that you use orthogonal initialization with parameter $\sqrt{2}$ for kernels and initialize biases with zeros.

In [3]:
N_ACTIONS = 8


In [4]:
import torch
import torch.nn as nn
import torch.functional as F


class DQN(nn.Module):
	def __init__(self, N_actions, input_size=1568, hidden_size=448):
		super(DQN, self).__init__()
		self.conv1 = nn.Conv2d(4, 16, kernel_size=5, stride=2)
		self.bn1 = nn.BatchNorm2d(16)
		self.relu1 = nn.ReLU()
		self.conv2 = nn.Conv2d(16, 32, kernel_size=5, stride=2)
		self.bn2 = nn.BatchNorm2d(32)
		self.relu2 = nn.ReLU()

		self.conv3 = nn.Conv2d(32, 32, kernel_size=5, stride=2)
		self.bn3 = nn.BatchNorm2d(32)
		self.relu3 = nn.ReLU()

		self.neck = nn.Linear(input_size, hidden_size)
		self.relu4 = nn.ReLU()
		self.action_head = nn.Linear(hidden_size, N_actions)
		self.value_head = nn.Linear(hidden_size, 1)

	def forward(self, x):
		x = self.relu1(self.bn1(self.conv1(x)))
		x = self.relu2(self.bn2(self.conv2(x)))
		x = self.relu3(self.bn3(self.conv3(x)))
		x = x.view(x.size(0), -1)
		x = self.relu4(self.neck(x))
		return self.action_head(x), self.value_head(x)

You will also need to define and use a policy that wraps the model. While the model computes logits for all actions, the policy will sample actions and also compute their log probabilities.  `policy.act` should return a dictionary of all the arrays that are needed to interact with an environment and train the model.
 Note that actions must be an `np.ndarray` while the other
tensors need to have the type determined by your deep learning framework.

In [5]:
class Policy:
    def __init__(self, model):
        self.model = model

    def act(self, inputs):
        # Implement a policy by calling the model, sampling actions and computing their log probs.
        # Should return a dict containing keys ["actions", "logits", "log_probs", "values"].
        with torch.no_grad():
            output = {}
            logits, values = self.model(torch.tensor(inputs))
            action_probs = torch.softmax(logits, dim=-1)
            actions = np.array([np.random.choice(N_ACTIONS, p=probs) for probs in action_probs.numpy()])
            log_probs = torch.log(action_probs)
            output["actions"] = actions
            output["logits"] = logits
            output["action_probs"] = action_probs
            output["log_probs"] = log_probs
            output["values"] = values
            return output

Next will pass the environment and policy to a runner that collects partial trajectories from the environment.
The class that does is is already implemented for you.

In [6]:
from runners import EnvRunner

This runner interacts with the environment for a given number of steps and returns a dictionary containing
keys

* "observations"
* "rewards"
* "resets"
* "actions"
* all other keys that you defined in `Policy`

under each of these keys there is a python `list` of interactions with the environment. This list has length $T$ that is size of partial trajectory. Partial trajectory for given moment `t` is part of `ComputeValueTargets.__call__` input argument `trajectory` from moment `t` to the end (i.e. it"s different at each iteration in the algorithm).

To train the part of the model that predicts state values you will need to compute the value targets.
Any callable could be passed to `EnvRunner` to be applied to each partial trajectory after it is collected.
Thus, we can implement and use `ComputeValueTargets` callable.
The formula for the value targets is simple:

$$
\hat v(s_t) = \left( \sum_{t"=0}^{T - 1} \gamma^{t"}r_{t+t"} \right) + \gamma^T \hat{v}(s_{t+T}),
$$

In implementation, however, do not forget to use
`trajectory["resets"]` flags to check if you need to add the value targets at the next step when
computing value targets for the current step. You can access `trajectory["state"]["latest_observation"]`
to get last observations in partial trajectory &mdash; $s_{t+T}$.

In [7]:
class ComputeValueTargets:
    def __init__(self, policy, gamma=0.99):
        self.policy = policy
        self.gamma = gamma

    def __call__(self, trajectory):
        """Compute value targets for a given partial trajectory."""

        rewards = trajectory["rewards"]
        resets = trajectory["resets"]
        
        
        value_targets = np.zeros_like(rewards)
        
        T = len(rewards)
        
        # We"ll use a flag to indicate the "episode boundary" after the reset
        for t in reversed(range(T)):
            if resets[t]:
                # At a reset, start a new value target calculation from this point
                sm = rewards[t]
                value_targets[t] = sm
            else:
                if t == T - 1:
                    # For the last step in the trajectory (end of this episode), just use the reward
                    sm = self.policy(trajectory["state"]["latest_observation"])[1]
                    value_targets[t] = sm
                else:
                    # Add the immediate reward + discounted value of the next step
                    sm = sm * self.gamma + rewards[t]
                    value_targets[t] = sm

        trajectory["value_targets"] = value_targets

After computing value targets we will transform lists of interactions into tensors
with the first dimension `batch_size` which is equal to `env_steps * num_envs`, i.e. you essentially need
to flatten the first two dimensions.

In [8]:
class MergeTimeBatch:
    """ Merges first two axes typically representing time and env batch. """
    def __call__(self, trajectory):
        for key in ["resets", "rewards", "value_targets", "logits", "log_probs", "values", "action_probs"]:          
               trajectory[key] = trajectory[key].reshape((len(trajectory[key]) * len(len(trajectory[key])),
                                                          -1))


In [9]:
model = DQN(N_actions=N_ACTIONS)
policy = Policy(model)
runner = EnvRunner(
    env=env,
    policy=policy,
    nsteps=5,
    transforms=[
        ComputeValueTargets(policy),
        MergeTimeBatch(),
    ],
)


Now is the time to implement the advantage actor critic algorithm itself. You can look into your lecture,
[Mnih et al. 2016](https://arxiv.org/abs/1602.01783) paper, and [lecture](https://www.youtube.com/watch?v=Tol_jw5hWnI&list=PLkFD6_40KJIxJMR-j5A1mkxK26gh_qg37&index=20) by Sergey Levine.

In [10]:
a = torch.tensor([[1,2,3],
              [4,5,6]])
b = torch.tensor([[10], [20]])
torch.mean(a, dtype=torch.float)
a-torch.ones_like(a)*b

tensor([[ -9,  -8,  -7],
        [-16, -15, -14]])

In [11]:
class A2C:
    def __init__(self,
                 policy,
                 optimizer,
                 value_loss_coef=0.25,
                 entropy_coef=0.01,
                 max_grad_norm=0.5):
        self.policy = policy
        self.optimizer = optimizer
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.max_grad_norm = max_grad_norm
        self.lr_scheduler=None

    def policy_loss(self, trajectory):
        # You will need to compute advantages here.
        log_probs_for_actions = trajectory["log_probs"]
        advantages = trajectory["value_targets"]-torch.ones_like(trajectory["value_targets"]) * trajectory["values"]
        loss = torch.sum(log_probs_for_actions*advantages, dim=-1)
        return loss

    def value_loss(self, trajectory):
        loss = torch.mean((trajectory["values"] - trajectory["value_targets"]) ** 2)
        return loss
    
    def loss(self, trajectory):
        entropy = -self.entropy_coef * torch.sum(trajectory["log_probs"]*trajectory["log_probs"]) # max
        value_loss = self.value_loss(trajectory) # min
        policy_loss = self.policy_loss(trajectory) # maximize
        return -entropy + value_loss - policy_loss

    def step(self, trajectory):
        # Compute loss here. Don"t forgen entropy regularization with `entropy_coef` 
        
        # Gradient descent step
        loss = self.loss(trajectory)
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(policy.model.parameters(), self.max_grad_norm)
        self.optimizer.step()
        self.optimizer.zero_grad()

        if self.lr_scheduler:
            self.lr_scheduler.step()
        return (loss,
                grad_norm,
                self.policy_loss(trajectory),
                torch.sum(trajectory['rewards'])
                )

Now you can train your model. With reasonable hyperparameters training on a single GTX1080 for 10 million steps across all batched environments (which translates to about 5 hours of wall clock time)
it should be possible to achieve *average raw reward over last 100 episodes* (the average is taken over 100 last
episodes in each environment in the batch) of about 600. You should plot this quantity with respect to
`runner.step_var` &mdash; the number of interactions with all environments. It is highly
encouraged to also provide plots of the following quantities (these are useful for debugging as well):

* [Coefficient of Determination](https://en.wikipedia.org/wiki/Coefficient_of_determination) between
value targets and value predictions
* Entropy of the policy $\pi$
* Value loss
* Policy loss
* Value targets
* Value predictions
* Gradient norm
* Advantages
* A2C loss

For optimization we suggest you use RMSProp with learning rate starting from 7e-4 and linearly decayed to 0, smoothing constant (alpha in PyTorch and decay in TensorFlow) equal to 0.99 and epsilon equal to 1e-5.

In [12]:
def lr_lambda(iter):
    return max(0, 1 - iter / 1e+7)

optimizer = torch.optim.RMSprop(model.parameters(), lr=7e-4, alpha=0.99, eps=1e-5)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

a2c = A2C(policy, optimizer)

In [13]:
from tqdm.notebook import trange


step = 0
total_steps = 10e+6 // nenvs
log_freq = 100

with trange(step, int(total_steps + 1)) as progress_bar:
    for step in progress_bar:
        trajectory = runner.get_next()
        loss, grad_norm, p3, p4 = a2c.step(trajectory)

        if step % log_freq == 0:
            print(step)

            clear_output(True)


  0%|          | 0/1250001 [00:00<?, ?it/s]

_____ 0
1
2
3 [4 3 4 6 3 3 3 4]


EOFError: 

### Target networks?

You may recall a technique called "target networks" we used a few weeks ago when we trained a DQN agent to play Atari Breakout and wonder why we have not suggested using them here. The answer is that this is more historical than practical.

While the "chasing the target" problem is still present in actor-critic value estimation and target networks do show up in follow-up papers, the original A3C/A2C papers do not mention them and do not explain this omission.

The hypothesis why this may not be a big deal (compared to Q-learning) goes like this. An A3C/A2C agent selects actions based on policy, not an epsilon greedy exploration function, for which the argmax can change drastically due to tiny errors in function approximation. Therefore, errors in the value target caused by target chasing will cause less damage.

Also, the actor-critic gradient relies on the advantage function $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$. Compare this to the $Q$-function $Q(s_t, a_t) = r(s_t, a_t) + \gamma \cdot \mathbb{E}_{s_{t+1} \mid s_t, a_t} V(s_{t+1})$ used in Q-learning and SARSA: we would expect that any bias in $V$-function approximation will be carried over from $V(s_{t+1})$ to $V(s_t)$ by gradient updates. However, in the formula for the advantage function the two approximations ($Q$-function and $V$-function) come with opposite signs, and thus the errors will cancel out.

The last reason may be computational. Authors were concerned to beat existent algorithms in the wall-clock learning time, and any overhead of parameter copying (target network update) counted against this goal.